# Phase 2: Hybrid Weapon Detection Training 

Colab training pipeline.

| Component | Location | Speed |
|-----------|----------|-------|
| Source Code & Model Weights | Google Drive (persistent) | — |
| Dataset (41k images, 10 GB) | `yolo_dataset.zip` on GDrive → extracted to Colab SSD | ⚡ ~100 MB/s |
| Training I/O | Colab local SSD (`/content/`) | ⚡ Native |
| Checkpoints | Google Drive `models/weights/` | Auto-saved |

> **Run cells in order.** After Step 1, restart the session if prompted by the Numpy warning.

## Step 1 — Environment & Dependencies
Install core packages and fix the **Numpy < 2.0** binary incompatibility with Ultralytics on T4.

In [ ]:
# ── Install packages ────────────────────────────────────────────────────────
%pip install -q ultralytics albumentations timm

# ── Force Numpy < 2.0 (binary ABI compatibility with Ultralytics on T4) ─────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.0"], check=True)

# ── Verify ───────────────────────────────────────────────────────────────────
import numpy as np, torch, os, sys
from pathlib import Path

print(f"✅ Numpy  : {np.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — check Runtime type!'}")

if np.__version__.startswith('2.'):
    print("\n⚠️  [ACTION REQUIRED] Numpy 2.x still present.")
    print("    Click 'RESTART SESSION' in the Colab popup, then re-run from Step 2.")
else:
    print("\n✅ Environment ready. Proceed to Step 2.")

## Step 2 — Mount Google Drive + Extract Dataset

### Prerequisites
Before running this cell, you must have:
1. Created `yolo_dataset.zip` from your local `data/processed/yolo_dataset/` folder
2. Uploaded it to Google Drive (root or any folder)

Update `ZIP_GDRIVE_PATH` below to match where you placed the ZIP.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              USER-EDITABLE CONFIGURATION — change these paths            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys, time, shutil
from pathlib import Path
from google.colab import drive

# ── Where is yolo_dataset.zip on your Google Drive? ─────────────────────
ZIP_GDRIVE_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/yolo_dataset.zip"

# ── Where is your project source code on Google Drive? ──────────────────
GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/"

# ── Local extraction target (Colab SSD — fastest possible I/O) ──────────
LOCAL_DATASET_DIR = "/content/yolo_dataset"

# ═══════════════════════════════════════════════════════════════════════════
# ── 1. Mount Google Drive ─────────────────────────────────────────────────
drive.mount('/content/drive')

# ── 2. Align project root (source code + weights) ────────────────────────
PROJECT_ROOT = Path(GDRIVE_PROJECT_PATH)
if PROJECT_ROOT.exists():
    os.chdir(str(PROJECT_ROOT))
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"✅ Project root: {PROJECT_ROOT}")
else:
    raise FileNotFoundError(
        f"❌ Project root not found at:\n   {PROJECT_ROOT}\n"
        "   Update GDRIVE_PROJECT_PATH in this cell."
    )

# ── 3. Import model (confirms sys.path is correct) ───────────────────────
from models.hybrid_model import HybridWeaponDetector
print("✅ HybridWeaponDetector imported.")

# ── 4. Extract dataset to Colab SSD ──────────────────────────────────────
DATA_YAML_PATH = Path(LOCAL_DATASET_DIR) /"yolo_dataset"/"data.yaml"

if DATA_YAML_PATH.exists():
    print(f"✅ Dataset already extracted at {LOCAL_DATASET_DIR} — skipping unzip.")
else:
    # Verify ZIP exists
    if not os.path.exists(ZIP_GDRIVE_PATH):
        raise FileNotFoundError(
            f"❌ ZIP not found at: {ZIP_GDRIVE_PATH}\n"
            "   Upload yolo_dataset.zip to your Google Drive and update ZIP_GDRIVE_PATH."
        )
    
    zip_size_gb = os.path.getsize(ZIP_GDRIVE_PATH) / (1024**3)
    print(f"📦 Found ZIP: {ZIP_GDRIVE_PATH} ({zip_size_gb:.1f} GB)")
    
    # Copy ZIP from GDrive to local SSD first (faster than unzipping over FUSE)
    LOCAL_ZIP = "/content/yolo_dataset.zip"
    if not os.path.exists(LOCAL_ZIP):
        print("📋 Copying ZIP to Colab SSD (internal Google transfer)...")
        t0 = time.time()
        shutil.copy2(ZIP_GDRIVE_PATH, LOCAL_ZIP)
        dt = time.time() - t0
        print(f"Done in {dt:.0f}s ({zip_size_gb/dt*1024 if dt > 0 else 0:.0f} MB/s)")
    else:
        print("📋 ZIP already on local SSD — skipping copy.")
    
    # Unzip
    print("📂 Extracting dataset...")
    t0 = time.time()
    os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
    # -q for quiet, -o for overwrite (safe if partially extracted)
    !unzip -qo "{LOCAL_ZIP}" -d /content/
    dt = time.time() - t0
    print(f" Extracted in {dt:.0f}s")
    
    # Clean up local ZIP copy to save SSD space
    if os.path.exists(LOCAL_ZIP):
        os.remove(LOCAL_ZIP)
        print(" Cleaned up local ZIP copy.")

# ── 5. Final check ────────────────────────────────────────────────────────
if DATA_YAML_PATH.exists():
    print(f"\n✅ Dataset ready at: {LOCAL_DATASET_DIR}")
    print(f"✅ data.yaml: {DATA_YAML_PATH}")
else:
    raise FileNotFoundError(
        f"❌ data.yaml not found after extraction at {DATA_YAML_PATH}\n"
        "   The ZIP structure may be wrong. Expected: yolo_dataset/data.yaml"
)

## Step 3 — Verify Dataset
Confirm data.yaml is correct and the image counts match expectations.

In [ ]:
import yaml

with open(DATA_YAML_PATH) as f:
    cfg = yaml.safe_load(f)

print("═" * 50)
print("data.yaml")
print("═" * 50)
print(f"  nc    : {cfg.get('nc')}")
print(f"  names : {cfg.get('names')}")
print(f"  train : {cfg.get('train')}")
print(f"  val   : {cfg.get('val')}")
print(f"  test  : {cfg.get('test', 'N/A')}")
print("═" * 50)

dataset_root = DATA_YAML_PATH.parent
total = 0
for split in ['train', 'val', 'test']:
    img_dir = dataset_root / split / 'images'
    lbl_dir = dataset_root / split / 'labels'
    if img_dir.exists():
        n_img = len(list(img_dir.iterdir()))
        n_lbl = len(list(lbl_dir.iterdir())) if lbl_dir.exists() else 0
        total += n_img
        status = '✅' if n_img > 0 and n_lbl > 0 else '⚠️'
        print(f"{status} {split:5s}: {n_img:>6,} images | {n_lbl:>6,} labels")
    else:
        print(f"⚠️  {split:5s}: directory not found")

print(f"\n📊 Total images: {total:,}")
if total > 40000:
    print("✅ Dataset looks complete!")
else:
    print("⚠️  Image count lower than expected (41k+). Check your ZIP.")

## Step 4 — Hybrid Trainer (Two-Phase Schedule)

| Phase | Epochs | Backbone | Purpose |
|-------|--------|----------|---------|
| **1** | 1–10 | ❄️ Frozen | Stabilise neck + head feature alignment |
| **2** | 11–50 | 🔥 Unfrozen | Full architectural fine-tuning |

**Red Alert Config**: Class weight `[1.0, 1.0, 2.5]` — **2.5× focal penalty on Confuser** to suppress false positives.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from tqdm import tqdm
from pathlib import Path


# ── DataLoader Factory ───────────────────────────────────────────────────────
def get_dataloaders(data_yaml_path: str, batch_size: int = 16, imgsz: int = 640):
    """Build train/val DataLoaders from data.yaml."""
    data_cfg = check_det_dataset(data_yaml_path)
    train_set = YOLODataset(
        img_path=data_cfg['train'], imgsz=imgsz,
        augment=True, batch_size=batch_size, task='detect', data=data_cfg
    )
    val_set = YOLODataset(
        img_path=data_cfg['val'], imgsz=imgsz,
        augment=False, batch_size=batch_size, task='detect', data=data_cfg
    )
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, collate_fn=train_set.collate_fn
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True, collate_fn=val_set.collate_fn
    )
    print(f"✅ DataLoaders: {len(train_set):,} train / {len(val_set):,} val samples")
    return train_loader, val_loader


# ── Two-Phase Trainer ────────────────────────────────────────────────────────
class HybridTrainer:
    """
    Two-phase training with auto-resume and persistent checkpointing.
    
    Phase 1 (epochs 1-freeze_epochs): Backbone frozen, train neck + head only.
    Phase 2 (remaining epochs):       Backbone unfrozen, full fine-tuning.
    
    Checkpoints are saved to Google Drive for persistence across Colab sessions.
    """

    def __init__(self, model, train_loader, val_loader,
                 device="cuda", weights_dir="models/weights", freeze_epochs=10):
        self.device        = device
        self.model         = model.to(device)
        self.train_loader  = train_loader
        self.val_loader    = val_loader
        self.scaler        = GradScaler()
        self.weights_dir   = Path(weights_dir)
        self.freeze_epochs = freeze_epochs
        self.best_loss     = float('inf')
        self.weights_dir.mkdir(parents=True, exist_ok=True)
        self.criterion     = model.head.compute_loss

    def _set_backbone_frozen(self, frozen: bool):
        for param in self.model.backbone.parameters():
            param.requires_grad = not frozen
        tag = "frozen ❄️" if frozen else "unfrozen 🔥"
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"  Backbone {tag} | Trainable params: {trainable:,}")

    def _build_optimizer(self):
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        return optim.AdamW(trainable, lr=1e-4, weight_decay=1e-4)

    def save_checkpoint(self, optimizer, epoch, is_best=False, tag=None):
        state = {
            'epoch':                epoch,
            'model_state_dict':     self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss':            self.best_loss,
        }
        torch.save(state, self.weights_dir / "last.pt")
        if is_best:
            torch.save(self.model.state_dict(), self.weights_dir / "best.pt")
            print(f"  💾 best.pt updated (loss={self.best_loss:.4f})")
        if tag:
            torch.save(state, self.weights_dir / f"epoch_{tag}.pt")

    def load_checkpoint(self, optimizer):
        """Resume from last.pt if it exists on GDrive."""
        ckpt_path = self.weights_dir / "last.pt"
        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location=self.device)
            self.model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            self.best_loss = ckpt.get('best_loss', float('inf'))
            start = ckpt['epoch'] + 1
            print(f"✅ Resumed from epoch {ckpt['epoch']} (best_loss={self.best_loss:.4f})")
            return start
        return 1

    def train_epoch(self, optimizer, epoch, total_epochs):
        self.model.train()
        total_loss = 0.0
        phase = "Phase 1 ❄️" if epoch <= self.freeze_epochs else "Phase 2 🔥"
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]")
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            optimizer.zero_grad()
            with autocast():
                preds = self.model(imgs)
                loss  = self.criterion(preds, batch, self.device)
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return total_loss / len(self.train_loader)

    def run(self, total_epochs=50, resume=True):
        """Run two-phase training with auto-resume."""
        # Phase 1 setup
        self._set_backbone_frozen(True)
        optimizer = self._build_optimizer()
        start_epoch = 1

        if resume:
            start_epoch = self.load_checkpoint(optimizer)

        phase2_started = (start_epoch > self.freeze_epochs)
        if phase2_started:
            print("Resuming in Phase 2 — unfreezing backbone.")
            self._set_backbone_frozen(False)
            optimizer = self._build_optimizer()
            self.load_checkpoint(optimizer)

        for epoch in range(start_epoch, total_epochs + 1):
            # Phase transition at freeze_epochs + 1
            if epoch == self.freeze_epochs + 1 and not phase2_started:
                print("\n═══ Phase 2: Unfreezing backbone for full fine-tune ═══")
                self._set_backbone_frozen(False)
                optimizer = self._build_optimizer()
                phase2_started = True

            avg_loss = self.train_epoch(optimizer, epoch, total_epochs)
            is_best  = avg_loss < self.best_loss
            if is_best:
                self.best_loss = avg_loss

            tag = str(epoch) if epoch % 5 == 0 else None
            self.save_checkpoint(optimizer, epoch, is_best=is_best, tag=tag)
            print(f"Epoch {epoch:3d}/{total_epochs} | avg_loss={avg_loss:.4f} | best={self.best_loss:.4f}")

        print(f"\n🏁 Training complete. Checkpoints at: {self.weights_dir}")

print("✅ Trainer defined.")

## Step 5 — Launch Training

Set `resume=True` to auto-continue from `last.pt` if Colab disconnected.
Set `resume=False` for a fresh training run.

In [ ]:
# ── Device ───────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU! Go to Runtime > Change runtime type > T4 GPU")

# ── Model ────────────────────────────────────────────────────────────────────
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", nc=3, device=device)

# ── Red Alert: 2.5× focal penalty on Confuser (class 2) ─────────────────────
model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)
print("✅ Class weights: Weapon=1.0 | Person=1.0 | Confuser=2.5")

# ── DataLoaders ──────────────────────────────────────────────────────────────
train_loader, val_loader = get_dataloaders(
    str(DATA_YAML_PATH),
    batch_size=16,
    imgsz=640,
)

# ── Trainer (checkpoints to GDrive for persistence) ──────────────────────────
WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"
trainer = HybridTrainer(
    model         = model,
    train_loader  = train_loader,
    val_loader    = val_loader,
    device        = device,
    weights_dir   = str(WEIGHTS_DIR),
    freeze_epochs = 10,
)

# ── Launch ───────────────────────────────────────────────────────────────────
trainer.run(total_epochs=50, resume=True)